# EXPLORE COLONIAL COLLECTION OF THE WERELD MUSEUM
In order to run the code below please create a new virtual enviroment (I would not select your global pytohn installation) and intall the preprequesites from the following cell. You should be ready to go

In [3]:
## Prerequisites 
!pip install sparqlwrapper pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 12.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 10.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sparqlwrapper]0m [sparqlwrapper]


In [16]:
### Import and set the SPARQL endpoint
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON
from IPython.display import HTML, display

def show_image(url):
    if pd.notna(url):
        return HTML(f'<img src="{url}" width="150" />')
    return ""

# Make DataFrame display wide enough for long values
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)
pd.set_option("display.expand_frame_repr", False)

# 1. Set your endpoint URL here

SPARQL_ENDPOINT = "https://api.colonialcollections.nl/datasets/nmvw/collection-archives/sparql" 

## Explore costumes by their cultural function
The URI <https://hdl.handle.net/20.500.11840/termmaster10069572> points at concept "costume by function (English term)". 
The following query returns all the sub-types of costume by function and how many artifact belong to each type. 
NB. The * also allows to take into account they narrower concepts.

In [ ]:

query = """
PREFIX la: <https://linked.art/ns/terms/>
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?o (GROUP_CONCAT(DISTINCT ?o1; separator=" | ") AS ?labels) (COUNT(DISTINCT ?artifact) AS ?totalObjectCount) WHERE {
  <https://hdl.handle.net/20.500.11840/termmaster10069572> skos:narrower ?o.
  ?o skos:altLabel ?o1 .
  
  ?o skos:narrower* ?subConcept .
  OPTIONAL {
    ?artifact crm:P2_has_type ?subConcept .
  }
}
GROUP BY ?o
LIMIT 1000
"""

# 3. Set up the connection
sparql = SPARQLWrapper(SPARQL_ENDPOINT)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)

try:
    # 4. Execute query and convert to JSON
    results = sparql.query().convert()
    
    # 5. Parse JSON results into a clean Pandas DataFrame
    bindings = results["results"]["bindings"]
    
    # Extract just the values from the SPARQL JSON structure
    data = []
    for row in bindings:
        data.append({var: row[var]["value"] for var in row})
        
    df = pd.DataFrame(data)
    
    # Optional: Convert the count column to a numeric type
    if "totalObjectCount" in df.columns:
        df["totalObjectCount"] = pd.to_numeric(df["totalObjectCount"])
        
    # Display the first few rows of the DataFrame
    print(f"Successfully retrieved {len(df)} rows.")
    display(df)

except Exception as e:
    print(f"An error occurred: {e}")

Successfully retrieved 14 rows.


,o,labels,totalObjectCount
0,https://hdl.handle.net/20.500.11840/termmaster10069573,sports clothing,25
1,https://hdl.handle.net/20.500.11840/termmaster10069575,mourning dress | rouwdrachten,133
2,https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),236
3,https://hdl.handle.net/20.500.11840/termmaster10069581,deads garments,7
4,https://hdl.handle.net/20.500.11840/termmaster10069582,nachtkleden | night and dressing wear,22
5,https://hdl.handle.net/20.500.11840/termmaster10069588,bathing suits,2
6,https://hdl.handle.net/20.500.11840/termmaster10069589,werkkleding | work clothes,7
7,https://hdl.handle.net/20.500.11840/termmaster10069590,dress (worn in particular situations),420
8,https://hdl.handle.net/20.500.11840/termmaster10069595,ambtskledij | habits,51
9,https://hdl.handle.net/20.500.11840/termmaster10069596,uniforms,71


### Fetch the artifacts
Given all the subtypes of costumes, I can go ahead and pick one, and paste the URI in the query as chosenCategory.
Then is possible to retrieve all the artifacts belonging to that category. 
The label of the cathegories are both in english and dutch and they are concatenated in the same value in by the query.
Not all objects have a picutre, and some have more (that is why some rows are duplicated). Paste the Url in the browser to visualize the desired one, not all the links might be working. 

In [ ]:
query = """ PREFIX la: <https://linked.art/ns/terms/>
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX dct: <http://purl.org/dc/terms/>
SELECT  
  ?chosenCategory
  (GROUP_CONCAT(DISTINCT ?categoryLabelRaw; separator=" | ") AS ?chosenCategoryLabel) 
  ?specificType 
  (GROUP_CONCAT(DISTINCT ?typeLabelRaw; separator=" | ") AS ?typeLabel)
  ?artifact 
  ?artifacttitle
  ?url_photo 
  WHERE {
  
  BIND(<https://hdl.handle.net/20.500.11840/termmaster10069577> AS ?chosenCategory)
  ?chosenCategory skos:altLabel ?categoryLabelRaw .

  # 2. Recursively find the category itself and all of its narrower concepts
  ?chosenCategory skos:narrower* ?specificType .
  
  # 3. Find the objects that have these types
  ?artifact crm:P2_has_type ?specificType .


  # 4. Get labels and title for the objects and types
  OPTIONAL { 
    ?specificType skos:altLabel ?typeLabelRaw . 
  }
  OPTIONAL { 
    ?artifact dct:title ?artifacttitle . 
  }
  OPTIONAL {
    ?artifact a crm:E22_Human-Made_Object.
    ?artifact crm:P65_shows_visual_item ?vi.
    ?vi la:digitally_shown_by ?o2.
    ?o2 la:access_point ?url_photo .
  }
}
GROUP BY ?chosenCategory ?specificType ?artifact ?artifacttitle ?url_photo
LIMIT 1000
"""
# 3. Set up the connection
sparql = SPARQLWrapper(SPARQL_ENDPOINT)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)

try:
    # 4. Execute query and convert to JSON
    results = sparql.query().convert()
    
    # 5. Parse JSON results into a clean Pandas DataFrame
    bindings = results["results"]["bindings"]
    
    # Extract just the values from the SPARQL JSON structure
    data = []
    for row in bindings:
        data.append({var: row[var]["value"] for var in row})
        
    df = pd.DataFrame(data)

    def make_image_html(url):
        if pd.notna(url) and str(url).strip():
            return f'<img src="{url}" width="150" />'
        return ""

    df["photo"] = df["url_photo"].apply(make_image_html)

    html_table = df[["artifacttitle", "photo"]].to_html(escape=False, index=False)
    display(HTML(html_table))

except Exception as e:
    print(f"An error occurred: {e}")

Successfully retrieved 343 rows.


,chosenCategory,chosenCategoryLabel,specificType,typeLabel,artifact,artifacttitle,url_photo,photo
0,https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/1221247,VERKLEEDKLEDING (INDIAANS KOSTUUM),https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/RV//RV-08-36b_D061915.jpg,<IPython.core.display.HTML object>
1,https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/1221247,VERKLEEDKLEDING (INDIAANS KOSTUUM),https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/RV//RV-08-36b.jpg,<IPython.core.display.HTML object>
2,https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/1221247,VERKLEEDKLEDING (INDIAANS KOSTUUM),https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/RV//ScanNr\J8002\8002439b.jpg,<IPython.core.display.HTML object>
3,https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/1221248,VERKLEEDKLEDING (INDIAANS KOSTUUM),https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/RV//ScanNr\J8002\8002439c.jpg,<IPython.core.display.HTML object>
4,https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/1221248,VERKLEEDKLEDING (INDIAANS KOSTUUM),https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/RV//RV-08-36c.jpg,<IPython.core.display.HTML object>
...,...,...,...,...,...,...,...,...
338,https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/termmaster10069579,stage costumes | toneelkleren,https://hdl.handle.net/20.500.11840/681226,NaN,NaN,
339,https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/termmaster10069579,stage costumes | toneelkleren,https://hdl.handle.net/20.500.11840/681243,NaN,NaN,
340,https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/termmaster10069579,stage costumes | toneelkleren,https://hdl.handle.net/20.500.11840/681268,NaN,NaN,
341,https://hdl.handle.net/20.500.11840/termmaster10069577,dressing-up costumes | kostuums (verkleedkleding),https://hdl.handle.net/20.500.11840/termmaster10069579,stage costumes | toneelkleren,https://hdl.handle.net/20.500.11840/681287,NaN,NaN,


## Know the geographic provenance of the object


In [10]:
query = """
PREFIX la: <https://linked.art/ns/terms/>
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX dct: <http://purl.org/dc/terms/>

SELECT DISTINCT ?artifact ?artifactTitle ?productionEvent ?productionPlace ?placeLabel WHERE {
  BIND ( <https://hdl.handle.net/20.500.11840/1115435> as ?artifact)
  # 1. Target human-made objects in the collection
  ?artifact a crm:E22_Human-Made_Object .
  
  # 2. Get artifact titles if available
  OPTIONAL { 
    ?artifact dct:title ?artifactTitle . 
  }
  
  # 3. Follow the provenance path to the production event and location
  ?artifact crm:P108i_was_produced_by ?productionEvent .
  ?productionEvent crm:P7_took_place_at ?productionPlace .
  
  # 4. Get the human-readable name of the location
  ?productionPlace skos:prefLabel ?placeLabel .

}
LIMIT 1000
 """

# 3. Set up the connection
sparql = SPARQLWrapper(SPARQL_ENDPOINT)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)

try:
    # 4. Execute query and convert to JSON
    results = sparql.query().convert()
    
    # 5. Parse JSON results into a clean Pandas DataFrame
    bindings = results["results"]["bindings"]
    
    # Extract just the values from the SPARQL JSON structure
    data = []
    for row in bindings:
        data.append({var: row[var]["value"] for var in row})
        
    df = pd.DataFrame(data)
  
        
    # Display the first few rows of the DataFrame
    print(f"Successfully retrieved {len(df)} rows.")
    display(df)
except Exception as e:
    print(f"An error occurred: {e}")

Successfully retrieved 1 rows.


,artifact,artifactTitle,productionEvent,productionPlace,placeLabel
0,https://hdl.handle.net/20.500.11840/1115435,PINANGSCHAAR,https://data.colonialcollections.nl/nmvw/.well-known/genid/3fd62c2ea0adcd1abf0ba5e598441eb5,https://hdl.handle.net/20.500.11840/termmaster10063351,Bali


## Interesting links

### HOLDERS

https://hdl.handle.net/20.500.11840/termmaster10070376, namely "houders naar functie of context" refers to containers of various use. 
Use this uri to fire the same query from before.

In [14]:

query = """
PREFIX la: <https://linked.art/ns/terms/>
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?o (GROUP_CONCAT(DISTINCT ?o1; separator=" | ") AS ?labels) (COUNT(DISTINCT ?artifact) AS ?totalObjectCount) WHERE {
  <https://hdl.handle.net/20.500.11840/termmaster10070376> skos:narrower ?o.
  ?o skos:altLabel ?o1 .
  
  ?o skos:narrower* ?subConcept .
  OPTIONAL {
    ?artifact crm:P2_has_type ?subConcept .
  }
}
GROUP BY ?o
LIMIT 1000
"""

# 3. Set up the connection
sparql = SPARQLWrapper(SPARQL_ENDPOINT)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)

try:
    # 4. Execute query and convert to JSON
    results = sparql.query().convert()
    
    # 5. Parse JSON results into a clean Pandas DataFrame
    bindings = results["results"]["bindings"]
    
    # Extract just the values from the SPARQL JSON structure
    data = []
    for row in bindings:
        data.append({var: row[var]["value"] for var in row})
        
    df = pd.DataFrame(data)
    
    # Optional: Convert the count column to a numeric type
    if "totalObjectCount" in df.columns:
        df["totalObjectCount"] = pd.to_numeric(df["totalObjectCount"])
        
    # Display the first few rows of the DataFrame
    print(f"Successfully retrieved {len(df)} rows.")
    display(df)

except Exception as e:
    print(f"An error occurred: {e}")

Successfully retrieved 25 rows.


,o,labels,totalObjectCount
0,https://hdl.handle.net/20.500.11840/termmaster10070377,wrappers (containers),68
1,https://hdl.handle.net/20.500.11840/termmaster10070388,culinary containers,12414
2,https://hdl.handle.net/20.500.11840/termmaster10070595,containers for weapons,2728
3,https://hdl.handle.net/20.500.11840/termmaster10070606,containers for writing equipment,244
4,https://hdl.handle.net/20.500.11840/termmaster10070624,containers for textiles and needlework,129
5,https://hdl.handle.net/20.500.11840/termmaster10070635,housekeeping containers,105
6,https://hdl.handle.net/20.500.11840/termmaster10070644,gluepots,2
7,https://hdl.handle.net/20.500.11840/termmaster10070645,storage containers,1059
8,https://hdl.handle.net/20.500.11840/termmaster10070666,containers for personal use,7827
9,https://hdl.handle.net/20.500.11840/termmaster10070762,odorant burners,282


## CERIMONIAL OBJECTS

https://hdl.handle.net/20.500.11840/termmaster10068882 for cerimonial objects


https://hdl.handle.net/20.500.11840/termmaster10068883

In [15]:
query = """ PREFIX la: <https://linked.art/ns/terms/>
PREFIX crm: <http://www.cidoc-crm.org/cidoc-crm/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX dct: <http://purl.org/dc/terms/>
SELECT  
  ?chosenCategory
  (GROUP_CONCAT(DISTINCT ?categoryLabelRaw; separator=" | ") AS ?chosenCategoryLabel) 
  ?specificType 
  (GROUP_CONCAT(DISTINCT ?typeLabelRaw; separator=" | ") AS ?typeLabel)
  ?artifact 
  ?artifacttitle
  ?url_photo 
  WHERE {
  
  BIND(<https://hdl.handle.net/20.500.11840/termmaster10068883> AS ?chosenCategory)
  ?chosenCategory skos:altLabel ?categoryLabelRaw .

  # 2. Recursively find the category itself and all of its narrower concepts
  ?chosenCategory skos:narrower* ?specificType .
  
  # 3. Find the objects that have these types
  ?artifact crm:P2_has_type ?specificType .


  # 4. Get labels and title for the objects and types
  OPTIONAL { 
    ?specificType skos:altLabel ?typeLabelRaw . 
  }
  OPTIONAL { 
    ?artifact dct:title ?artifacttitle . 
  }
  OPTIONAL {
    ?artifact a crm:E22_Human-Made_Object.
    ?artifact crm:P65_shows_visual_item ?vi.
    ?vi la:digitally_shown_by ?o2.
    ?o2 la:access_point ?url_photo .
  }
}
GROUP BY ?chosenCategory ?specificType ?artifact ?artifacttitle ?url_photo
LIMIT 1000
"""
# 3. Set up the connection
sparql = SPARQLWrapper(SPARQL_ENDPOINT)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)

try:
    # 4. Execute query and convert to JSON
    results = sparql.query().convert()
    
    # 5. Parse JSON results into a clean Pandas DataFrame
    bindings = results["results"]["bindings"]
    
    # Extract just the values from the SPARQL JSON structure
    data = []
    for row in bindings:
        data.append({var: row[var]["value"] for var in row})
        
    df = pd.DataFrame(data)
  
        
    # Display the first few rows of the DataFrame
    print(f"Successfully retrieved {len(df)} rows.")
    display(df)
except Exception as e:
    print(f"An error occurred: {e}")

Successfully retrieved 34 rows.


,chosenCategory,chosenCategoryLabel,specificType,typeLabel,artifact,artifacttitle,url_photo
0,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/45370,EFFIGIE VAN SANDELHOUT VOORSTELLEND EEN VROUW,https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/TM//TM-1103-26a.jpg
1,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/45370,EFFIGIE VAN SANDELHOUT VOORSTELLEND EEN VROUW,https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/TM-Extra//TM-1103-26a_D041145.jpg
2,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/45371,WITTE KATOENEN LAP,https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/TM//TM-1103-26a.jpg
3,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/45380,HOUTEN EFFIGIEPLANK,https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/TM//TM-1103-30.jpg
4,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/45380,HOUTEN EFFIGIEPLANK,https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/TM-Extra//TM-1103-30_D004057.jpg
5,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/45381,BLIKKEN SCHAAL VOOR HET WATER DAT GEBRUIKT WORDT BIJ DE LIJKWASSING,https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/TM//TM-1103-31.jpg
6,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/45382,BLIKKEN SCHAAL VOOR HET WATER DAT GEBRUIKT WORDT BIJ DE LIJKWASSING,https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/TM//TM-1103-32.jpg
7,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/528127,Grafpot,https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/AM//AM-392-1.jpg
8,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/60162,LICHTGEEL HOUTEN EFFIGIEPLANK,https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/TM-Extra//TM-1646-10_D004063.jpg
9,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/termmaster10068883,funerary objects,https://hdl.handle.net/20.500.11840/60162,LICHTGEEL HOUTEN EFFIGIEPLANK,https://collectie.wereldculturen.nl/cc/imageproxy.ashx?server=localhost&port=17581&filename=images//Images/TM//TM-1646-10.jpg
